# LeetCode 3432 — Count Partitions with Even Sum Difference bằng Python

Link đề: https://leetcode.com/problems/count-partitions-with-even-sum-difference/ · Độ khó: **Easy** · Chủ đề: Array (mảng), Math (toán), Prefix Sum (tổng tiền tố), Parity (chẵn/lẻ)

Skeleton LeetCode:

```python
class Solution:
    def countPartitions(self, nums: List[int]) -> int:
        ...
```

Ghi chú: ba ví dụ chính lấy từ đề; mọi đáp án trong notebook đã được kiểm chứng bằng code (so sánh năm cách làm với nhau và với vét cạn theo định nghĩa, kể cả vét cạn **toàn bộ** mảng nhỏ). Ví dụ nào do mình tự dựng sẽ ghi rõ "tự dựng" — nếu cần khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một **phân hoạch (partition)** là chỉ số `i` (`0 <= i < n - 1`) chia mảng thành **hai mảng con không rỗng**: trái `nums[0..i]` và phải `nums[i+1..n-1]`; đếm số phân hoạch mà **hiệu** `tổng trái − tổng phải` là **số chẵn**.
2. Nắm các công cụ Python: `sum` và cắt lát `nums[:k]`, `nums[k:]`, **chẵn/lẻ** bằng `% 2` (kể cả với số âm), `enumerate`, tổng tiền tố bằng `itertools.accumulate`, phép **XOR** `^` và AND `&` để làm việc với bit chẵn/lẻ.
3. Đi từ cách **tính lại hai tổng cho từng phân hoạch** → **tổng trái/phải cập nhật dần (`l`, `r`)** → **mảng tiền tố** → **nhận xét toán học về tính chẵn/lẻ** → **chỉ đếm số phần tử lẻ (bit chẵn/lẻ bằng XOR)**.
4. Hiểu mấu chốt toán học: `trái − phải = trái − (tổng − trái) = 2·trái − tổng`. Vì `2·trái` luôn **chẵn**, nên **tính chẵn/lẻ của hiệu = tính chẵn/lẻ của `tổng`**, **không phụ thuộc** vị trí cắt. Do đó đáp án chỉ có hai khả năng: `n − 1` (nếu tổng chẵn) hoặc `0` (nếu tổng lẻ).
5. Tránh các lỗi kinh điển: đếm cả `i = n − 1` (phải rỗng), trả `n` thay vì `n − 1`, nhầm "hiệu chẵn" với "hai tổng cùng chẵn", và quên rằng `% 2` của số âm trong Python vẫn cho `0` hoặc `1` (nhưng ở C++/Java có thể ra `-1`).

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` có độ dài `n`.

Một **phân hoạch** là chỉ số `i` với `0 <= i < n - 1`, chia mảng thành hai mảng con **không rỗng**:

- Mảng trái gồm các chỉ số `[0, i]`.
- Mảng phải gồm các chỉ số `[i + 1, n - 1]`.

Trả về **số phân hoạch** mà **hiệu** giữa **tổng mảng trái** và **tổng mảng phải** là **số chẵn**.

**Ví dụ 1**

- Input: `nums = [10,10,3,7,6]`
- Output: `4`
- Giải thích: có 4 phân hoạch:
  - `[10]`, `[10, 3, 7, 6]`: hiệu `10 - 26 = -16`, chẵn.
  - `[10, 10]`, `[3, 7, 6]`: hiệu `20 - 16 = 4`, chẵn.
  - `[10, 10, 3]`, `[7, 6]`: hiệu `23 - 13 = 10`, chẵn.
  - `[10, 10, 3, 7]`, `[6]`: hiệu `30 - 6 = 24`, chẵn.

**Ví dụ 2**

- Input: `nums = [1,2,2]`
- Output: `0`
- Giải thích: không phân hoạch nào cho hiệu chẵn.

**Ví dụ 3**

- Input: `nums = [2,4,6,8]`
- Output: `3`
- Giải thích: mọi phân hoạch đều cho hiệu chẵn.

**Ràng buộc**

- `2 <= n == nums.length <= 100`
- `1 <= nums[i] <= 100`

In [1]:
from typing import List

examples = [
    ([10, 10, 3, 7, 6], 4),
    ([1, 2, 2], 0),
    ([2, 4, 6, 8], 3),
]

for nums, expected in examples:
    print("nums =", nums, "-> kỳ vọng", expected)

nums = [10, 10, 3, 7, 6] -> kỳ vọng 4
nums = [1, 2, 2] -> kỳ vọng 0
nums = [2, 4, 6, 8] -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Phân hoạch tại chỉ số `i` và cắt lát

Cắt lát `nums[:k]` lấy `k` phần tử đầu (chỉ số `0..k-1`); `nums[k:]` lấy phần còn lại (từ chỉ số `k`). Phân hoạch tại `i` có trái = `nums[:i + 1]` và phải = `nums[i + 1:]`. Vì phải **không rỗng**, `i` chỉ chạy từ `0` đến `n - 2`, tức `range(n - 1)`.

In [2]:
nums = [10, 10, 3, 7, 6]
n = len(nums)
for i in range(n - 1):                      # i = 0..n-2
    print(f"i={i}: trái = {nums[:i + 1]}, phải = {nums[i + 1:]}")
print("số phân hoạch =", n - 1)

i=0: trái = [10], phải = [10, 3, 7, 6]
i=1: trái = [10, 10], phải = [3, 7, 6]
i=2: trái = [10, 10, 3], phải = [7, 6]
i=3: trái = [10, 10, 3, 7], phải = [6]
số phân hoạch = 4


### 3.2. Chẵn / lẻ và phép `% 2`

`x % 2 == 0` nghĩa là `x` chẵn; `x % 2 == 1` nghĩa là `x` lẻ. Trong Python, `%` với số chia dương **luôn cho kết quả không âm**: `(-16) % 2 == 0`, `(-3) % 2 == 1`. (Ở C++/Java, `-3 % 2 == -1`; vì thế người ta hay kiểm tra `x % 2 != 0` hoặc `(x & 1)`.) Ta cần điều này vì **hiệu có thể âm** (`10 - 26 = -16`).

In [3]:
for x in [-16, -3, 0, 4, 7]:
    print(f"x={x}: x % 2 = {x % 2}, x & 1 = {x & 1}, chẵn? {x % 2 == 0}")

x=-16: x % 2 = 0, x & 1 = 0, chẵn? True
x=-3: x % 2 = 1, x & 1 = 1, chẵn? False
x=0: x % 2 = 0, x & 1 = 0, chẵn? True
x=4: x % 2 = 0, x & 1 = 0, chẵn? True
x=7: x % 2 = 1, x & 1 = 1, chẵn? False


### 3.3. Quy tắc chẵn/lẻ của phép cộng, trừ, nhân đôi

| phép tính | kết quả |
|---|---|
| chẵn ± chẵn | chẵn |
| lẻ ± lẻ | chẵn |
| chẵn ± lẻ | lẻ |
| `2 × bất_kỳ_số_nguyên` | **luôn chẵn** |

Điều quan trọng cho bài này: **`a − b` và `a + b` luôn cùng tính chẵn/lẻ** (vì `(a + b) − (a − b) = 2b` chẵn). Ta sẽ dùng chính tính chất này.

In [4]:
for a, b in [(10, 26), (23, 13), (7, 2), (3, 8)]:
    print(f"a={a}, b={b}: a-b={a - b} ({'chẵn' if (a - b) % 2 == 0 else 'lẻ'}), a+b={a + b} ({'chẵn' if (a + b) % 2 == 0 else 'lẻ'})")

a=10, b=26: a-b=-16 (chẵn), a+b=36 (chẵn)
a=23, b=13: a-b=10 (chẵn), a+b=36 (chẵn)
a=7, b=2: a-b=5 (lẻ), a+b=9 (lẻ)
a=3, b=8: a-b=-5 (lẻ), a+b=11 (lẻ)


### 3.4. Cập nhật tổng dần: `l += x`, `r -= x`

Thay vì tính lại tổng mỗi lần, ta giữ `l` (tổng trái) và `r` (tổng phải), ban đầu `l = 0`, `r = sum(nums)`. Khi "chuyển" phần tử `x` từ phải sang trái: `l += x` và `r -= x`. Đây là kỹ thuật "**hai tổng trượt**" rất thường gặp.

In [5]:
nums = [10, 10, 3, 7, 6]
l, r = 0, sum(nums)
print(f"bắt đầu: l={l}, r={r}")
for x in nums[:-1]:
    l += x
    r -= x
    print(f"chuyển {x}: l={l}, r={r}, l-r={l - r}")

bắt đầu: l=0, r=36
chuyển 10: l=10, r=26, l-r=-16
chuyển 10: l=20, r=16, l-r=4
chuyển 3: l=23, r=13, l-r=10
chuyển 7: l=30, r=6, l-r=24


### 3.5. Tổng tiền tố với `accumulate`

`accumulate(nums, initial=0)` cho `s` với `s[k]` = tổng `k` phần tử đầu. Tổng trái của phân hoạch `i` là `s[i + 1]`, tổng toàn mảng là `s[n]`, nên tổng phải là `s[n] - s[i + 1]`.

In [6]:
from itertools import accumulate

nums = [10, 10, 3, 7, 6]
s = list(accumulate(nums, initial=0))
print("s =", s)                                      # [0, 10, 20, 23, 30, 36]
n = len(nums)
for i in range(n - 1):
    left, right = s[i + 1], s[n] - s[i + 1]
    print(f"i={i}: trái={left}, phải={right}, hiệu={left - right}")

s = [0, 10, 20, 23, 30, 36]
i=0: trái=10, phải=26, hiệu=-16
i=1: trái=20, phải=16, hiệu=4
i=2: trái=23, phải=13, hiệu=10
i=3: trái=30, phải=6, hiệu=24


### 3.6. Bit chẵn/lẻ: `x & 1` và `^` (XOR)

`x & 1` lấy **bit cuối** của `x`: `0` nếu chẵn, `1` nếu lẻ. `a ^ b` (XOR) cho `1` khi hai bit **khác nhau**, `0` khi giống nhau; vì thế `p ^= (x & 1)` qua nhiều `x` cho **tính chẵn/lẻ của số phần tử lẻ** (tức tính chẵn/lẻ của tổng). Ta sẽ dùng ở Cách 5.

In [7]:
for a in (0, 1):
    for b in (0, 1):
        print(f"{a} ^ {b} = {a ^ b}")
p = 0
for x in [10, 10, 3, 7, 6]:
    p ^= x & 1
    print(f"sau {x}: p = {p}")
print("tổng =", sum([10, 10, 3, 7, 6]), "-> tổng % 2 =", sum([10, 10, 3, 7, 6]) % 2, "| p =", p)

0 ^ 0 = 0
0 ^ 1 = 1
1 ^ 0 = 1
1 ^ 1 = 0
sau 10: p = 0
sau 10: p = 0
sau 3: p = 1
sau 7: p = 0
sau 6: p = 0
tổng = 36 -> tổng % 2 = 0 | p = 0


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `nums = [10, 10, 3, 7, 6]` (tổng `36`). Liệt kê **mọi phân hoạch** và hiệu:

| `i` | trái | phải | tổng trái | tổng phải | hiệu trái − phải | chẵn? |
|---|---|---|---|---|---|---|
| 0 | `[10]` | `[10,3,7,6]` | 10 | 26 | −16 | có |
| 1 | `[10,10]` | `[3,7,6]` | 20 | 16 | 4 | có |
| 2 | `[10,10,3]` | `[7,6]` | 23 | 13 | 10 | có |
| 3 | `[10,10,3,7]` | `[6]` | 30 | 6 | 24 | có |

Cả `4` phân hoạch đều chẵn. Hãy thử `nums = [1, 2, 2]` (tổng `5`): `i=0`: `1 − 4 = −3` (lẻ); `i=1`: `3 − 2 = 1` (lẻ) ⇒ `0` phân hoạch.

**Quan sát:** ở ví dụ đầu **mọi** phân hoạch đều chẵn, ở ví dụ sau **không** phân hoạch nào chẵn. Chẳng lẽ kết quả không phụ thuộc vào vị trí cắt? Ta chứng minh ở mục 6.

In [8]:
for nums in ([10, 10, 3, 7, 6], [1, 2, 2], [2, 4, 6, 8]):
    n = len(nums)
    print(f"nums = {nums}, tổng = {sum(nums)}")
    for i in range(n - 1):
        left, right = sum(nums[:i + 1]), sum(nums[i + 1:])
        print(f"   i={i}: {left} - {right} = {left - right} -> {'chẵn' if (left - right) % 2 == 0 else 'lẻ'}")

nums = [10, 10, 3, 7, 6], tổng = 36
   i=0: 10 - 26 = -16 -> chẵn
   i=1: 20 - 16 = 4 -> chẵn
   i=2: 23 - 13 = 10 -> chẵn
   i=3: 30 - 6 = 24 -> chẵn
nums = [1, 2, 2], tổng = 5
   i=0: 1 - 4 = -3 -> lẻ
   i=1: 3 - 2 = 1 -> lẻ
nums = [2, 4, 6, 8], tổng = 20
   i=0: 2 - 18 = -16 -> chẵn
   i=1: 6 - 14 = -8 -> chẵn
   i=2: 12 - 8 = 4 -> chẵn


## 5. Cách 1 — Brute Force / baseline: tính lại hai tổng cho từng phân hoạch

**Ý tưởng.** Làm đúng như bảng tính tay: với mỗi `i` từ `0` đến `n - 2`, tính `left = sum(nums[:i + 1])` và `right = sum(nums[i + 1:])`, rồi kiểm tra `(left - right) % 2 == 0`; nếu đúng thì tăng biến đếm.

In [9]:
def count_partitions_brute(nums: List[int]) -> int:
    n = len(nums)
    count = 0
    for i in range(n - 1):                       # phải không rỗng nên dừng ở n - 2
        left = sum(nums[:i + 1])
        right = sum(nums[i + 1:])
        if (left - right) % 2 == 0:
            count += 1
    return count


for nums, expected in examples:
    print(count_partitions_brute(nums), "(kỳ vọng", expected, ")")

4 (kỳ vọng 4 )
0 (kỳ vọng 0 )
3 (kỳ vọng 3 )


**Độ phức tạp (Cách 1).** Mỗi `i` tính hai tổng, mỗi tổng `O(n)` ⇒ **Time `O(n²)`**, **Space `O(n)`** (cắt lát tạo list). Với `n <= 100` thì quá đủ nhanh.

**Nút thắt:** hai tổng ở `i + 1` chỉ khác `i` đúng một phần tử `nums[i + 1]`. Không cần tính lại từ đầu — cập nhật dần.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — cập nhật dần:** giữ `l` và `r`, mỗi bước chuyển một phần tử từ phải sang trái. Tổng `O(n)`.

**Bước 2 — nhận xét toán học (bước đột phá thật sự).** Gọi `T` là tổng toàn mảng, `L` là tổng trái. Khi đó tổng phải là `T − L` và

`hiệu = L − (T − L) = 2L − T`.

`2L` luôn **chẵn**, nên `2L − T` có **cùng tính chẵn/lẻ với `T`** (chẵn − chẵn = chẵn; chẵn − lẻ = lẻ). Vậy:

- Nếu `T` **chẵn** ⇒ mọi phân hoạch đều cho hiệu chẵn ⇒ đáp án là **tất cả `n − 1` phân hoạch**.
- Nếu `T` **lẻ** ⇒ không phân hoạch nào chẵn ⇒ đáp án `0`.

Vị trí cắt **không ảnh hưởng**. Bài này thực chất chỉ hỏi: "tổng toàn mảng chẵn hay lẻ?".

In [10]:
# Kiểm chứng đẳng thức hiệu = 2L - T trên ví dụ 1
nums = [10, 10, 3, 7, 6]
T = sum(nums)
L = 0
for i in range(len(nums) - 1):
    L += nums[i]
    hieu_truc_tiep = L - (T - L)
    hieu_cong_thuc = 2 * L - T
    print(f"i={i}: L={L}, T-L={T - L}, hiệu={hieu_truc_tiep}, 2L-T={hieu_cong_thuc}, cùng chẵn/lẻ với T={T}? {hieu_cong_thuc % 2 == T % 2}")
    assert hieu_truc_tiep == hieu_cong_thuc

i=0: L=10, T-L=26, hiệu=-16, 2L-T=-16, cùng chẵn/lẻ với T=36? True
i=1: L=20, T-L=16, hiệu=4, 2L-T=4, cùng chẵn/lẻ với T=36? True
i=2: L=23, T-L=13, hiệu=10, 2L-T=10, cùng chẵn/lẻ với T=36? True
i=3: L=30, T-L=6, hiệu=24, 2L-T=24, cùng chẵn/lẻ với T=36? True


## 7. Các cách tiếp theo

### Cách 2 — Hai tổng trượt `l`, `r` (một lượt duyệt)

**Giải thích từng đoạn.**

1. `l = 0`, `r = sum(nums)`: lúc đầu trái rỗng, phải là cả mảng.
2. `for x in nums[:-1]`: chỉ duyệt đến phần tử **kế cuối** vì phần phải phải **không rỗng** (`nums[:-1]` bỏ phần tử cuối).
3. Mỗi bước: `l += x`, `r -= x` (chuyển `x` sang trái); nếu `(l - r) % 2 == 0` thì `count += 1`.

In [11]:
def count_partitions_running(nums: List[int]) -> int:
    l, r = 0, sum(nums)
    count = 0
    for x in nums[:-1]:                          # bỏ phần tử cuối để phải không rỗng
        l += x
        r -= x
        if (l - r) % 2 == 0:
            count += 1
    return count


for nums, expected in examples:
    print(count_partitions_running(nums), "(kỳ vọng", expected, ")")

4 (kỳ vọng 4 )
0 (kỳ vọng 0 )
3 (kỳ vọng 3 )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionRunning:
    def countPartitions(self, nums: List[int]) -> int:
        l, r = 0, sum(nums)
        ans = 0
        for x in nums[:-1]:
            l += x
            r -= x
            if (l - r) % 2 == 0:
                ans += 1
        return ans


print(SolutionRunning().countPartitions([10, 10, 3, 7, 6]))      # 4

4


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(n)`** (`nums[:-1]` tạo bản sao; dùng `range(n - 1)` để được `O(1)`).

### Cách 3 — Mảng tiền tố `s` rồi kiểm tra `2·s[i+1] − s[n]`

**Giải thích.** Dựng `s = accumulate(nums, initial=0)`. Với mỗi `i` trong `range(n - 1)`: tổng trái `s[i + 1]`, tổng phải `s[n] - s[i + 1]`; kiểm tra hiệu chẵn. Cách này dùng lại ý tưởng tiền tố quen thuộc và dễ mở rộng sang truy vấn khác.

In [13]:
def count_partitions_prefix(nums: List[int]) -> int:
    n = len(nums)
    s = list(accumulate(nums, initial=0))
    count = 0
    for i in range(n - 1):
        left = s[i + 1]
        right = s[n] - s[i + 1]
        if (left - right) % 2 == 0:
            count += 1
    return count


for nums, expected in examples:
    print(count_partitions_prefix(nums), "(kỳ vọng", expected, ")")

4 (kỳ vọng 4 )
0 (kỳ vọng 0 )
3 (kỳ vọng 3 )


In [14]:
# Code nộp LeetCode cho Cách 3
from itertools import accumulate


class SolutionPrefix:
    def countPartitions(self, nums: List[int]) -> int:
        n = len(nums)
        s = list(accumulate(nums, initial=0))
        return sum((s[i + 1] - (s[n] - s[i + 1])) % 2 == 0 for i in range(n - 1))


print(SolutionPrefix().countPartitions([1, 2, 2]))               # 0

0


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`**.

### Cách 4 — Nhận xét toán học: đáp án là `n − 1` hoặc `0`

**Giải thích.** Từ mục 6: `hiệu = 2L − T` cùng tính chẵn/lẻ với `T`. Vậy chỉ cần `sum(nums) % 2`: nếu `0` trả `n − 1`, nếu `1` trả `0`. Không cần vòng lặp qua các phân hoạch.

In [15]:
def count_partitions_math(nums: List[int]) -> int:
    return len(nums) - 1 if sum(nums) % 2 == 0 else 0


for nums, expected in examples:
    print(count_partitions_math(nums), "(kỳ vọng", expected, ")")

4 (kỳ vọng 4 )
0 (kỳ vọng 0 )
3 (kỳ vọng 3 )


In [16]:
# Code nộp LeetCode cho Cách 4
class SolutionMath:
    def countPartitions(self, nums: List[int]) -> int:
        return 0 if sum(nums) % 2 else len(nums) - 1


print(SolutionMath().countPartitions([2, 4, 6, 8]))              # 3

3


**Độ phức tạp Cách 4:** **Time `O(n)`** (một lần `sum`), **Space `O(1)`**.

### Cách 5 — Chỉ cần bit chẵn/lẻ: XOR các bit cuối

**Giải thích.** Tính chẵn/lẻ của tổng chỉ phụ thuộc vào **số phần tử lẻ**: có **số lẻ các số lẻ** ⇒ tổng lẻ; **số chẵn các số lẻ** ⇒ tổng chẵn. XOR dồn các bit `x & 1` cho đúng bit chẵn/lẻ này. Cách này không bao giờ tạo ra số lớn (hữu ích khi tổng thật có thể tràn kiểu số ở ngôn ngữ khác).

In [17]:
def count_partitions_parity_xor(nums: List[int]) -> int:
    parity = 0
    for x in nums:
        parity ^= x & 1                          # parity = (số phần tử lẻ) mod 2
    return 0 if parity else len(nums) - 1


for nums, expected in examples:
    print(count_partitions_parity_xor(nums), "(kỳ vọng", expected, ")")

4 (kỳ vọng 4 )
0 (kỳ vọng 0 )
3 (kỳ vọng 3 )


In [18]:
# Code nộp LeetCode cho Cách 5
class SolutionParityXor:
    def countPartitions(self, nums: List[int]) -> int:
        parity = 0
        for x in nums:
            parity ^= x & 1
        return 0 if parity else len(nums) - 1


print(SolutionParityXor().countPartitions([10, 10, 3, 7, 6]))    # 4

4


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(1)`**.

Bản nộp mình chọn là **Cách 4** (`sum(nums) % 2`): ngắn, `O(n)` thời gian, `O(1)` bộ nhớ và thể hiện đúng nhận xét toán học. Nếu bạn chưa tin nhận xét đó, **Cách 2** (hai tổng trượt) là bản "an toàn" bám sát đề — và ta sẽ kiểm chứng chéo ở mục test.

## 8. TRACE từng bước

Trace **Cách 2** (`l`, `r` trượt) trên `nums = [10, 10, 3, 7, 6]`, `T = 36`:

| bước | `x` chuyển | `l` | `r` | `l − r` | chẵn? | `count` |
|---|---|---|---|---|---|---|
| 0 (bắt đầu) | — | 0 | 36 | — | — | 0 |
| 1 | 10 | 10 | 26 | −16 | có | 1 |
| 2 | 10 | 20 | 16 | 4 | có | 2 |
| 3 | 3 | 23 | 13 | 10 | có | 3 |
| 4 | 7 | 30 | 6 | 24 | có | 4 |

Phần tử cuối `6` **không** được chuyển (nếu chuyển, phải sẽ rỗng). Đáp án `4`.

In [19]:
def trace_running(nums: List[int]) -> int:
    l, r = 0, sum(nums)
    count = 0
    print(f"nums = {nums}, bắt đầu l={l}, r={r}")
    for x in nums[:-1]:
        l += x
        r -= x
        even = (l - r) % 2 == 0
        if even:
            count += 1
        print(f"  chuyển x={x}: l={l}, r={r}, l-r={l - r}, {'chẵn' if even else 'lẻ'}, count={count}")
    return count


assert trace_running([10, 10, 3, 7, 6]) == 4
print()
assert trace_running([1, 2, 2]) == 0

nums = [10, 10, 3, 7, 6], bắt đầu l=0, r=36
  chuyển x=10: l=10, r=26, l-r=-16, chẵn, count=1
  chuyển x=10: l=20, r=16, l-r=4, chẵn, count=2
  chuyển x=3: l=23, r=13, l-r=10, chẵn, count=3
  chuyển x=7: l=30, r=6, l-r=24, chẵn, count=4

nums = [1, 2, 2], bắt đầu l=0, r=5
  chuyển x=1: l=1, r=4, l-r=-3, lẻ, count=0
  chuyển x=2: l=3, r=2, l-r=1, lẻ, count=0


Trace **Cách 4** (nhận xét toán học) trên cả ba ví dụ, in ra tổng, tính chẵn/lẻ và đáp án — để thấy đáp án chỉ phụ thuộc vào `T`.

In [20]:
def trace_math(nums: List[int]) -> int:
    T = sum(nums)
    n = len(nums)
    ans = n - 1 if T % 2 == 0 else 0
    print(f"nums={nums}: T={T} ({'chẵn' if T % 2 == 0 else 'lẻ'}), n-1={n - 1} -> đáp án {ans}")
    return ans


for nums, expected in examples:
    assert trace_math(nums) == expected

nums=[10, 10, 3, 7, 6]: T=36 (chẵn), n-1=4 -> đáp án 4
nums=[1, 2, 2]: T=5 (lẻ), n-1=2 -> đáp án 0
nums=[2, 4, 6, 8]: T=20 (chẵn), n-1=3 -> đáp án 3


## 9. Code nộp LeetCode

Bản nộp dùng Cách 4 (nhận xét toán học). Nếu muốn bản "không cần chứng minh", đổi thân hàm thành Cách 2.

In [21]:
class Solution:
    def countPartitions(self, nums: List[int]) -> int:
        # hiệu = trái - phải = 2*trái - tổng  ==> cùng chẵn/lẻ với tổng, không phụ thuộc vị trí cắt
        # tổng chẵn: cả n - 1 phân hoạch đều thỏa; tổng lẻ: không phân hoạch nào thỏa
        return 0 if sum(nums) % 2 else len(nums) - 1


sol = Solution()
for nums, expected in examples:
    print(sol.countPartitions(nums), "(kỳ vọng", expected, ")")

4 (kỳ vọng 4 )
0 (kỳ vọng 0 )
3 (kỳ vọng 3 )


## 10. Vì sao thuật toán đúng

- **Công thức hiệu:** với phân hoạch `i`, gọi `L` là tổng trái và `T` là tổng cả mảng; tổng phải là `T − L`. Hiệu `L − (T − L) = 2L − T`.
- **Tính chẵn/lẻ:** `2L` chẵn (mọi số nguyên nhân `2` đều chẵn) nên `2L − T ≡ −T ≡ T (mod 2)`. Chẵn/lẻ của `−T` bằng chẵn/lẻ của `T`.
- **Không phụ thuộc `i`:** vế phải chỉ chứa `T`, nên **tất cả** phân hoạch cùng chẵn hoặc cùng lẻ.
- **Số phân hoạch:** `i` chạy `0..n−2` nên có đúng `n − 1` phân hoạch (với `n >= 2` thì `n − 1 >= 1`).
- **Kết luận:** `T` chẵn ⇒ đáp án `n − 1`; `T` lẻ ⇒ đáp án `0`. Cách 1–3 (đếm trực tiếp) và Cách 4–5 (công thức) là cùng một đáp án, và ta kiểm chứng điều đó trên nhiều input ở mục 12.
- **Số âm:** dù `L − (T − L)` có thể âm, `% 2` của Python vẫn cho `0/1` đúng chẵn/lẻ, nên kiểm tra `== 0` hợp lệ.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)` (`n <= 100`).

| Cách | Time | Space |
|---|---|---|
| 1. Tính lại hai tổng mỗi phân hoạch | `O(n²)` | `O(n)` |
| 2. Hai tổng trượt `l`, `r` | `O(n)` | `O(n)` (hoặc `O(1)` nếu dùng `range(n - 1)`) |
| 3. Mảng tiền tố | `O(n)` | `O(n)` |
| 4. `sum(nums) % 2` | `O(n)` | `O(1)` |
| 5. XOR bit chẵn/lẻ | `O(n)` | `O(1)` |

Cách 4 và 5 không thể tốt hơn `O(n)` vì phải đọc từng phần tử ít nhất một lần.

## 12. Bộ test

Gom các cách vào `impls` rồi so với kỳ vọng. "Trọng tài" `oracle` đếm trực tiếp theo đúng định nghĩa bằng hai vòng `for` (không `sum`, không cắt lát).

In [22]:
import itertools
import random

sol = Solution()
impls = {
    "brute": count_partitions_brute,
    "running": count_partitions_running,
    "prefix": count_partitions_prefix,
    "math": count_partitions_math,
    "parity_xor": count_partitions_parity_xor,
    "leetcode": sol.countPartitions,
}


def oracle(nums):
    """Trọng tài: đếm đúng theo định nghĩa, tính hai tổng bằng vòng for."""
    n = len(nums)
    count = 0
    for i in range(n - 1):
        left = 0
        for j in range(0, i + 1):
            left += nums[j]
        right = 0
        for j in range(i + 1, n):
            right += nums[j]
        diff = left - right
        if diff % 2 == 0:
            count += 1
    return count


def check_all(nums, expected, label):
    for key, fn in impls.items():
        actual = fn(list(nums))
        assert actual == expected, (label, key, nums, actual, expected)
    assert oracle(nums) == expected, (label, "oracle")
    print("PASS", label)


for idx, (nums, expected) in enumerate(examples, 1):
    check_all(nums, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

Các ví dụ "tự dựng" dưới đây có kỳ vọng tính tay ngay trong comment.

In [23]:
check_all([1, 1], 1, "n=2, tổng 2 chẵn: 1 phân hoạch (1 - 1 = 0)")
check_all([1, 2], 0, "n=2, tổng 3 lẻ: 1 - 2 = -1 lẻ")
check_all([2, 2], 1, "n=2, hai số chẵn")
check_all([100, 100], 1, "n=2, giá trị lớn nhất")
check_all([1, 1, 1], 0, "tổng 3 lẻ: 1-2=-1, 2-1=1")
check_all([1, 1, 1, 1], 3, "tổng 4 chẵn: cả 3 phân hoạch (hiệu -2, 0, 2)")
check_all([1, 1, 1, 1, 2], 4, "tổng 6 chẵn: n - 1 = 4")
check_all([99, 100], 0, "tổng 199 lẻ")
check_all([100] * 100, 99, "n = 100, tổng chẵn: 99 phân hoạch")
check_all([1] * 100, 99, "n = 100, 100 số lẻ -> tổng 100 chẵn")
check_all([1] * 99 + [2], 0, "99 số lẻ + 1 chẵn: tổng 101 lẻ")

PASS n=2, tổng 2 chẵn: 1 phân hoạch (1 - 1 = 0)
PASS n=2, tổng 3 lẻ: 1 - 2 = -1 lẻ
PASS n=2, hai số chẵn
PASS n=2, giá trị lớn nhất
PASS tổng 3 lẻ: 1-2=-1, 2-1=1
PASS tổng 4 chẵn: cả 3 phân hoạch (hiệu -2, 0, 2)
PASS tổng 6 chẵn: n - 1 = 4
PASS tổng 199 lẻ
PASS n = 100, tổng chẵn: 99 phân hoạch
PASS n = 100, 100 số lẻ -> tổng 100 chẵn
PASS 99 số lẻ + 1 chẵn: tổng 101 lẻ


### 12.2. Phản ví dụ cho các cách làm sai

Hai cách sai thường gặp: (a) **đếm cả `i = n − 1`** (phải rỗng) bằng `range(n)`; (b) **đòi cả hai tổng đều chẵn** thay vì "hiệu chẵn".

In [24]:
def wrong_include_last(nums):
    l, r = 0, sum(nums)
    count = 0
    for x in nums:                               # SAI: duyệt cả phần tử cuối -> phải rỗng
        l += x
        r -= x
        if (l - r) % 2 == 0:
            count += 1
    return count


def wrong_both_even(nums):
    n = len(nums)
    return sum(1 for i in range(n - 1) if sum(nums[:i + 1]) % 2 == 0 and sum(nums[i + 1:]) % 2 == 0)   # SAI


print(wrong_include_last([2, 4, 6, 8]), "!=", sol.countPartitions([2, 4, 6, 8]))
print(wrong_both_even([1, 1]), "!=", sol.countPartitions([1, 1]))
assert wrong_include_last([2, 4, 6, 8]) != sol.countPartitions([2, 4, 6, 8])
assert wrong_both_even([1, 1]) != sol.countPartitions([1, 1])

4 != 3
0 != 1


### 12.3. Đối chiếu ngẫu nhiên (có seed) và vét cạn mảng nhỏ

(1) **Ngẫu nhiên:** `n` trong `[2, 100]`, giá trị trong `[1, 100]` — mọi cách và `oracle` phải cùng kết quả. (2) **Vét cạn:** mọi mảng độ dài `2..6` với giá trị trong `{1, 2, 3}` (tổng `3^2 + ... + 3^6 = 1089` mảng). (3) **Tính chất:** kết quả chỉ có thể là `0` hoặc `n − 1`.

In [25]:
rng = random.Random(3432)
for trial in range(2000):
    n = rng.randint(2, 100)
    nums = [rng.randint(1, 100) for _ in range(n)]
    ref = oracle(nums)
    for key, fn in impls.items():
        assert fn(list(nums)) == ref, (key, nums)
    assert ref in (0, n - 1)
print("PASS 2000 mảng ngẫu nhiên đúng ràng buộc")

count_arrays = 0
for n in range(2, 7):
    for nums in itertools.product([1, 2, 3], repeat=n):
        nums = list(nums)
        ref = oracle(nums)
        for key, fn in impls.items():
            assert fn(list(nums)) == ref, (key, nums)
        count_arrays += 1
print("PASS vét cạn", count_arrays, "mảng nhỏ")

PASS 2000 mảng ngẫu nhiên đúng ràng buộc
PASS vét cạn 1089 mảng nhỏ


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đếm cả `i = n − 1` (phần phải rỗng)

In [26]:
def sai_dem_ca_cuoi(nums):
    n = len(nums)
    count = 0
    for i in range(n):                              # SAI: i = n - 1 làm phải rỗng
        left = sum(nums[:i + 1])
        right = sum(nums[i + 1:])
        if (left - right) % 2 == 0:
            count += 1
    return count


print("sai:", sai_dem_ca_cuoi([2, 4, 6, 8]), "| đúng:", Solution().countPartitions([2, 4, 6, 8]))

sai: 4 | đúng: 3


**Vì sao sai:** đề yêu cầu **hai mảng con không rỗng**, nên `i` chỉ đến `n − 2`. Ở `i = n − 1`, mảng phải rỗng (tổng `0`) và hiệu bằng tổng cả mảng — có thể "tình cờ" chẵn và bị đếm thừa. **Cách sửa:** `range(n - 1)` hoặc `nums[:-1]`.

### Lỗi 2: Trả về `n` thay vì `n − 1` khi tổng chẵn

In [27]:
def sai_tra_n(nums):
    return len(nums) if sum(nums) % 2 == 0 else 0     # SAI: có n - 1 phân hoạch, không phải n


print("sai:", sai_tra_n([2, 4, 6, 8]), "| đúng:", Solution().countPartitions([2, 4, 6, 8]))

sai: 4 | đúng: 3


**Vì sao sai:** mảng `n` phần tử có `n − 1` vị trí cắt giữa các phần tử. **Cách sửa:** `len(nums) - 1`.

### Lỗi 3: Nhầm "hiệu chẵn" với "cả hai tổng chẵn"

In [28]:
def sai_hai_tong_chan(nums):
    n = len(nums)
    c = 0
    for i in range(n - 1):
        left, right = sum(nums[:i + 1]), sum(nums[i + 1:])
        if left % 2 == 0 and right % 2 == 0:            # SAI: hiệu chẵn còn xảy ra khi cả hai cùng lẻ
            c += 1
    return c


print("sai:", sai_hai_tong_chan([1, 1]), "| đúng:", Solution().countPartitions([1, 1]))

sai: 0 | đúng: 1


**Vì sao sai:** hiệu `a − b` chẵn khi `a` và `b` **cùng tính chẵn/lẻ** — cùng chẵn **hoặc cùng lẻ** (`1 − 1 = 0` chẵn). Chỉ kiểm tra "cùng chẵn" bỏ sót trường hợp cùng lẻ. **Cách sửa:** kiểm tra `(a - b) % 2 == 0` (hoặc `a % 2 == b % 2`).

### Lỗi 4: Kiểm tra "lẻ" bằng `== 1` khi hiệu có thể âm (an toàn trong Python, nguy hiểm ở ngôn ngữ khác)

In [29]:
print((-3) % 2, (-16) % 2)          # Python: 1 0  (luôn 0 hoặc 1)
# Ở C++/Java: -3 % 2 == -1, nên kiểm tra  diff % 2 == 1  sẽ SAI với hiệu âm.
# Cách an toàn mọi ngôn ngữ: kiểm tra "chẵn" bằng  diff % 2 == 0  hoặc  (diff & 1) == 0.
for diff in [-16, -3, 0, 7, 10]:
    print(diff, "chẵn?", diff % 2 == 0, "| & 1:", (diff & 1) == 0)

1 0
-16 chẵn? True | & 1: True
-3 chẵn? False | & 1: False
0 chẵn? True | & 1: True
7 chẵn? False | & 1: False
10 chẵn? True | & 1: True


**Vì sao cần để ý:** hiệu `L − R` có thể **âm** (ví dụ `10 − 26 = −16`). Python luôn trả phần dư không âm khi chia cho `2`, nhưng nhiều ngôn ngữ trả `-1` cho `-3 % 2`. Hãy quen kiểm tra **chẵn** bằng `== 0` (hoặc `& 1 == 0`) thay vì kiểm tra lẻ bằng `== 1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Hiệu của hai tổng** chẵn/lẻ" ⇒ nghĩ ngay đến **`a − b ≡ a + b (mod 2)`**: tính chẵn/lẻ của hiệu bằng tính chẵn/lẻ của **tổng cả hai**, tức tổng toàn mảng.
- Khi "**chia mảng thành hai phần**" và hỏi điều kiện trên `trái − phải`, viết `phải = T − trái` để chỉ còn **một** ẩn `trái`.
- Bài hỏi về **chẵn/lẻ** thường rút gọn thành "**đếm số phần tử lẻ**" hoặc "**XOR các bit cuối**".
- `n` phần tử có `n − 1` điểm cắt; **hai phần không rỗng** ⇒ `i` chạy `0..n−2`.
- Khi chưa chắc một nhận xét toán học, hãy **vét cạn mảng nhỏ** (như mục 12.3) để kiểm chứng trước khi tin.
- Trong Python `%` với số chia dương luôn cho `0/1`; ở ngôn ngữ khác hãy kiểm tra `== 0` thay vì `== 1`.

## 15. Biến thể của bài

1. **Hiệu chia hết cho `k`** (thay vì chẵn): `(2L − T) % k == 0`. Với `k` lẻ thì kết quả **phụ thuộc** vị trí cắt (không còn mẹo "mọi phân hoạch giống nhau").
2. **Đếm phân hoạch có `tổng trái > tổng phải`** (bài Number of Ways to Split Array - LeetCode 2270; chạy `l`, `r` trượt).
3. **Đếm phân hoạch mà hai tổng bằng nhau** — chỉ có thể xảy ra khi `T` chẵn và `L = T / 2` (Find Pivot Index - LeetCode 724, Partition Equal Subset Sum - LeetCode 416 là phiên bản khó hơn).
4. **Hiệu tích** hoặc **hiệu số phần tử chẵn** giữa hai phần.
5. **Trả về danh sách chỉ số** `i` thỏa điều kiện thay vì số lượng.

In [30]:
def count_diff_divisible_by_k(nums, k):
    """Biến thể 1: đếm phân hoạch có (trái - phải) % k == 0."""
    l, r = 0, sum(nums)
    count = 0
    for x in nums[:-1]:
        l += x
        r -= x
        if (l - r) % k == 0:
            count += 1
    return count


def count_left_greater(nums):
    """Biến thể 2: đếm phân hoạch có tổng trái > tổng phải."""
    l, r = 0, sum(nums)
    count = 0
    for x in nums[:-1]:
        l += x
        r -= x
        if l > r:
            count += 1
    return count


def indices_even_diff(nums):
    """Biến thể 5: danh sách các chỉ số i thỏa điều kiện."""
    T = sum(nums)
    return list(range(len(nums) - 1)) if T % 2 == 0 else []


print(count_diff_divisible_by_k([10, 10, 3, 7, 6], 2))      # k = 2 là bài gốc -> 4
print(count_diff_divisible_by_k([10, 10, 3, 7, 6], 3))      # hiệu -16, 4, 10, 24 -> chỉ 24 chia hết cho 3 -> 1
print(count_left_greater([10, 10, 3, 7, 6]))                # trái-phải: -16, 4, 10, 24 -> 3 giá trị dương
print(indices_even_diff([2, 4, 6, 8]), indices_even_diff([1, 2, 2]))
assert count_diff_divisible_by_k([10, 10, 3, 7, 6], 2) == 4
assert count_diff_divisible_by_k([10, 10, 3, 7, 6], 3) == 1
assert count_left_greater([10, 10, 3, 7, 6]) == 3
assert indices_even_diff([2, 4, 6, 8]) == [0, 1, 2]
assert indices_even_diff([1, 2, 2]) == []

4
1
3
[0, 1, 2] []


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tính lại hai tổng | `sum(nums[:i+1])`, `sum(nums[i+1:])` | `O(n²)` | `O(n)` | bám sát đề, làm trọng tài |
| 2. Hai tổng trượt `l`, `r` | chuyển một phần tử mỗi bước | `O(n)` | `O(n)` / `O(1)` | khi chưa nhận ra mẹo chẵn/lẻ; dùng được cho biến thể |
| 3. Mảng tiền tố | `s[i+1]`, `s[n] - s[i+1]` | `O(n)` | `O(n)` | khi cần nhiều truy vấn tổng đoạn |
| 4. `sum(nums) % 2` | hiệu cùng chẵn/lẻ với tổng | `O(n)` | `O(1)` | **nên nộp**: ngắn, nhanh |
| 5. XOR bit chẵn/lẻ | chỉ giữ bit cuối | `O(n)` | `O(1)` | tránh số lớn, dạng bài chẵn/lẻ tổng quát |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `7 % 2`, `(-4) % 2`, `5 & 1`, `1 ^ 1`, `Solution().countPartitions([1, 3, 5])`.

**Bài 2 (tính tay).** Với `nums = [3, 1, 4, 2]`: liệt kê ba phân hoạch, tính hiệu từng phân hoạch, xem chẵn hay lẻ, rồi đối chiếu với `sum(nums) % 2`.

**Bài 3 (điền TODO).** Hoàn thành `count_partitions_todo(nums)` theo Cách 2 (hai tổng trượt).

In [31]:
def count_partitions_todo(nums):
    # TODO 1: l = 0, r = sum(nums), count = 0
    # TODO 2: for x in nums[:-1]  (bỏ phần tử cuối để phải không rỗng)
    #   TODO 3: l += x; r -= x
    #   TODO 4: nếu (l - r) % 2 == 0 thì count += 1
    # TODO 5: return count
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `count_partitions_from_scratch(nums)` theo **Cách 5** (XOR) nhưng **không dùng** toán tử `^` hay `&`: đếm `odd = sum(1 for x in nums if x % 2 == 1)` rồi quyết định theo `odd % 2`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_partitions_diff_mod(nums, k)` đếm phân hoạch có hiệu chia hết cho `k` (với `k` bất kỳ), rồi kiểm tra: với `k = 2` ra kết quả như bài gốc, với `k = 3` thì kết quả **không** chỉ có thể là `0` hoặc `n − 1` (tìm một ví dụ). (b) Viết `first_even_partition(nums)` trả về chỉ số `i` **nhỏ nhất** thỏa điều kiện (hoặc `-1`) **mà không dùng vòng lặp**: chỉ dùng nhận xét toán học.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `1`, `0`, `1`, `0`; với `[1, 3, 5]` tổng `9` lẻ ⇒ `0`. Bài 3: dùng `nums[:-1]`.
- **Mức 2 (vừa):** Bài 2: tổng `10` chẵn. `i=0`: `3 − 7 = −4` (chẵn); `i=1`: `4 − 6 = −2` (chẵn); `i=2`: `8 − 2 = 6` (chẵn) ⇒ `3` phân hoạch. Bài 4: `odd % 2 == 0` ⇒ tổng chẵn.
- **Mức 3 (sâu):** Bài 5(a): tìm ví dụ với `k = 3` bằng cách thử mảng nhỏ (ví dụ `[1, 1, 1, 1]`: hiệu `−2, 0, 2`, chỉ `0` chia hết cho `3` ⇒ `1`, khác `0` và `n − 1 = 3`). 5(b): nếu `sum(nums) % 2 == 0` thì trả `0` (phân hoạch đầu tiên), ngược lại `-1`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert 7 % 2 == 1
assert (-4) % 2 == 0
assert 5 & 1 == 1
assert 1 ^ 1 == 0
assert Solution().countPartitions([1, 3, 5]) == 0

# Bài 2
nums2 = [3, 1, 4, 2]
hieu = [sum(nums2[:i + 1]) - sum(nums2[i + 1:]) for i in range(3)]
assert hieu == [-4, -2, 6]
assert all(h % 2 == 0 for h in hieu) and sum(nums2) % 2 == 0
assert Solution().countPartitions(nums2) == 3


# Bài 3
def count_partitions_todo_ans(nums):
    l, r, count = 0, sum(nums), 0
    for x in nums[:-1]:
        l += x
        r -= x
        if (l - r) % 2 == 0:
            count += 1
    return count


for nums, expected in examples:
    assert count_partitions_todo_ans(nums) == expected


# Bài 4
def count_partitions_from_scratch(nums):
    odd = sum(1 for x in nums if x % 2 == 1)
    return len(nums) - 1 if odd % 2 == 0 else 0


for nums, expected in examples:
    assert count_partitions_from_scratch(nums) == expected


# Bài 5
def count_partitions_diff_mod(nums, k):
    l, r, count = 0, sum(nums), 0
    for x in nums[:-1]:
        l += x
        r -= x
        if (l - r) % k == 0:
            count += 1
    return count


def first_even_partition(nums):
    return 0 if sum(nums) % 2 == 0 else -1


assert count_partitions_diff_mod([10, 10, 3, 7, 6], 2) == 4
assert count_partitions_diff_mod([1, 1, 1, 1], 3) == 1          # hiệu -2, 0, 2 -> chỉ 0 chia hết cho 3 (khác 0 và n - 1 = 3)
assert first_even_partition([2, 4, 6, 8]) == 0
assert first_even_partition([1, 2, 2]) == -1
print("Tất cả đáp án đều đúng")

Tất cả đáp án đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm phân hoạch có điều kiện chẵn/lẻ trên hiệu hai tổng)

```python
class Solution:
    def countPartitions(self, nums: List[int]) -> int:
        # hiệu = trái - phải = 2*trái - tổng  -> cùng chẵn/lẻ với tổng
        return 0 if sum(nums) % 2 else len(nums) - 1
```

Bản "không cần mẹo" (hai tổng trượt):

```python
l, r = 0, sum(nums)
ans = 0
for x in nums[:-1]:          # phải không rỗng
    l += x
    r -= x
    if (l - r) % 2 == 0:
        ans += 1
return ans
```

**Checklist**

- [ ] Số phân hoạch là `n − 1` (không phải `n`), mỗi phần **không rỗng**?
- [ ] Đã dùng được `phải = tổng − trái` và nhận ra hiệu `= 2·trái − tổng`?
- [ ] Kiểm tra **chẵn** bằng `% 2 == 0` (an toàn với hiệu âm)?
- [ ] Không nhầm "hiệu chẵn" với "hai tổng cùng chẵn"?
- [ ] Đã thử `n = 2`, tổng lẻ (đáp án `0`), tổng chẵn (đáp án `n − 1`)?
- [ ] Đã vét cạn mảng nhỏ hoặc đối chiếu với bản đếm trực tiếp?